<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Choose a model for hospital readmissions

**[Choose a model for hospital readmissions](https://learning.nextia-ai.com/courses/ml/m09/choose-a-model-for-readmissions/)** · Machine Learning: From Problem to Reliable Model · Module 9, case study 5 of 7 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** take a real readmission problem from raw data to a model that a care team can use: prepare the data, split it by patient, compare logistic regression, a decision tree, a random forest and gradient boosting under one protocol, and decide with a bootstrap interval and the one-standard-error rule whether the best score is worth its complexity.

| | |
|---|---|
| **Time** | About 120 minutes with the case-study page. The cells run in about 1 to 3 minutes in total. |
| **Needs** | An internet connection for the setup cells. No account. |
| **You should know** | Modules 1 to 8 of this course, especially [Tune hyperparameters](https://learning.nextia-ai.com/courses/ml/m06/tune-hyperparameters/#the-one-standard-error-rule), [Ensembles](https://learning.nextia-ai.com/courses/ml/m05/ensembles/#the-cost-of-complexity) and [When a simple model is better](https://learning.nextia-ai.com/courses/ml/m07/when-a-simple-model-is-better/). |
| **You do not need** | The Larkfield project. This notebook downloads its own data. |
| **Tested** | 2026-10-07, Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

The [case-study page](https://learning.nextia-ai.com/courses/ml/m09/choose-a-model-for-readmissions/) tells the full story, with the charts and the knowledge checks. This notebook runs every code block of the page, with short notes; read each output, then the **What this tells us** note under it.

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m09/choose-a-model-for-readmissions/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M09-L05-choose-a-model-for-readmissions/choose-a-model-for-readmissions-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the data

Run the next cell. It downloads the case-study data (about 3 MB) from the Nextia Learning labs repository into a folder `readmissions/data`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds. If a copy of a file is next to this notebook, the cell uses that copy.

You should see `Ready: diabetes_130_hospitals.csv.gz (3.0 MB), IDS_mapping.csv (0.0 MB).`

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `readmissions` (in Colab: the **Files** panel on the left), then run the cell again.

The data is real, de-identified hospital data: *Diabetes 130-US hospitals for years 1999–2008*, by Clore, Cios, DeShazo and Strack, UCI Machine Learning Repository, [doi:10.24432/C5230J](https://doi.org/10.24432/C5230J), licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). In our copy, the text `None` in the columns `A1Cresult` and `max_glu_serum` is an empty cell; nothing else is changed. The [dataset card](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/case-studies/data/diabetes_130_hospitals.md) has the details.

In [ ]:
# Setup: download the case-study data into readmissions/data/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

DATA_URL = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/case-studies/data/"
FILES = {
    "diabetes_130_hospitals.csv.gz": "7e2cd49bd158b99bef449897d3f5118881dc469115c6e80189fd248355b08eac",
    "IDS_mapping.csv": "f1bb82b471cb34649352597572c9b1fb00bd27f77b9f5a22a03dc3eb1039749e",
}
HERE = Path.cwd()
# The notebook works in its own folder readmissions/, marked with a file.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
FOLDER = Path("readmissions").resolve()
(FOLDER / "data").mkdir(parents=True, exist_ok=True)
(FOLDER / ".nextia-notebook").touch()


def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")


for name, expected in FILES.items():
    path = FOLDER / "data" / name
    if not (path.exists() and sha256(path) == expected):
        if (HERE / name).exists():
            shutil.copy(HERE / name, path)          # a copy next to the notebook
        else:
            download(DATA_URL + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder readmissions and run again.")

os.chdir(FOLDER)  # work in the case-study folder
print("Ready:", ", ".join(f"{name} ({(Path('data') / name).stat().st_size / 1e6:.1f} MB)" for name in FILES) + ".")

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd` and numpy as `np`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

## Setup: the experiment log

The next cell defines the helpers for your **experiment log**, the file `experiment_log.csv` in `readmissions/`, with the same columns as in [Machine learning is iterative](https://learning.nextia-ai.com/courses/ml/m05/machine-learning-is-iterative/#the-experiment-log).

- `log_experiment(id, change, fold_scores, valid_ap, decision, reason)` adds one row with today's date, the mean and the standard deviation of the 5 fold scores. If the id is already in the log, it replaces that row, so you can run a cell again. It also keeps the fold scores in the dictionary `FOLD_SCORES`.
- `show_log()` shows the log as a table.

Run the cell. It prints nothing.

In [ ]:
# The experiment log: one row per experiment, in experiment_log.csv (the C05 columns).
import datetime
from pathlib import Path

LOG_FILE = Path("experiment_log.csv")
LOG_COLUMNS = ["id", "date", "change", "cv_ap", "cv_std", "valid_ap", "decision", "reason"]
FOLD_SCORES = {}   # the 5 fold scores of each experiment, to compare models fold by fold


def log_experiment(id, change, fold_scores, valid_ap=None, decision="", reason=""):
    """Add one row to the log, or replace the row with the same id."""
    FOLD_SCORES[id] = np.asarray(fold_scores)
    log = pd.read_csv(LOG_FILE, dtype={"date": str}) if LOG_FILE.exists() else pd.DataFrame(columns=LOG_COLUMNS)
    row = {"id": id, "date": datetime.date.today().isoformat(), "change": change,
           "cv_ap": round(float(np.mean(fold_scores)), 3), "cv_std": round(float(np.std(fold_scores)), 3),
           "valid_ap": None if valid_ap is None else round(float(valid_ap), 3),
           "decision": decision, "reason": reason}
    rows = [r for r in log.to_dict("records") if r["id"] != id] + [row]
    pd.DataFrame(rows, columns=LOG_COLUMNS).sort_values("id").to_csv(LOG_FILE, index=False)


def show_log(columns=("id", "change", "cv_ap", "cv_std", "valid_ap", "decision")):
    """The log as a table (without the date and the reason, unless you ask for them)."""
    log = pd.read_csv(LOG_FILE, dtype={"date": str}).sort_values("id")
    with pd.option_context("display.max_colwidth", 70, "display.width", 200):
        print(log[list(columns)].fillna("").to_string(index=False))

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The situation

Nadia Haddad leads the care-transitions team of a hospital group. Her nurses phone patients in the first days after they leave hospital, to check medicines, book a follow-up visit and ask who helps at home. The team can call about **one discharged patient in ten**. Today, the nurses call the patients with the most hospital stays in the past year. Felix Brandt, the quality director, wants evidence: which model, if any, and is a complex model worth it?

A call to a patient who would not come back costs about 20 minutes of a nurse's time. A patient who comes back and was not called is the expensive error. The calls are fixed at 10%, so this is a ranking problem: who are the 10% most at risk? ([The situation on the page](https://learning.nextia-ai.com/courses/ml/m09/choose-a-model-for-readmissions/#the-situation).)

## 2. The data and first checks

The next cell loads the file. The source writes a missing value as `?`.

> **Check.** You should see `(101766, 50)`, 101,766 encounters, 71,518 patients and no duplicate rows.

In [ ]:
raw = pd.read_csv("data/diabetes_130_hospitals.csv.gz", na_values="?", low_memory=False)
print(raw.shape)
print("encounters:", raw["encounter_id"].nunique(), "| patients:", raw["patient_nbr"].nunique())
print("duplicate rows:", raw.duplicated().sum())

**What this tells us.** Every row is a different hospital stay, but there are only 71,518 patients: about 30,000 rows are later stays of the same people. This decides how we split the data.

The next cell shows the first rows of the columns that matter most. An empty test result shows as `NaN`.

In [ ]:
raw[["encounter_id", "patient_nbr", "age", "discharge_disposition_id", "time_in_hospital",
     "number_inpatient", "diag_1", "A1Cresult", "readmitted"]].head()

**What this tells us.** One row is one stay: the patient's number, the age band, a code for where the patient went, the stay's length, the hospital stays in the year before, the main diagnosis, a test result and the target `readmitted`.

The next cell shows the share of missing values in each column that has any.

> **Check.** You should see `weight` at 0.969 at the top and `diag_1` at 0.000 at the bottom.

In [ ]:
missing = raw.isna().mean()
missing[missing > 0].sort_values(ascending=False).round(3)

**What this tells us.** `weight` is almost always missing, so we drop it. An empty `A1Cresult` or `max_glu_serum` means "the test was not done": that is a value, not a gap. `medical_specialty` is missing in half of the stays, so "missing" becomes a value of its own.

The target column:

> **Predict.** What share of stays end with a return within 30 days (`<30`)?

In [ ]:
raw["readmitted"].value_counts()

**What this tells us.** Only 11,357 of 101,766 stays (11%) end with a return within 30 days. We predict `<30`; `>30` and `NO` count as "did not come back within 30 days". With a rare positive class, accuracy would mislead.

### Translate the codes

`IDS_mapping.csv` holds three code tables, one after the other, with an empty line between them. The next cell reads them into one dictionary per column.

> **Check.** You should see tables of 8, 30 and 25 codes, and the eight admission types.

In [ ]:
import csv

codes, column = {}, None
with open("data/IDS_mapping.csv", newline="") as file:
    for row in csv.reader(file):
        if not row or not row[0]:
            continue                          # the empty line between two tables
        if not row[0].isdigit():
            column = row[0]                   # a header line starts a new table
            codes[column] = {}
        else:
            codes[column][int(row[0])] = row[1].strip()
print({name: len(table) for name, table in codes.items()})
print(codes["admission_type_id"])

**What this tells us.** Three admission codes (`Not Available`, `NULL`, `Not Mapped`) all mean "we do not know". We will join them into one value, `unknown`.

### Remove the stays that cannot come back

Patients who died, or went to a hospice, cannot be readmitted. The next cell prints those six codes from the table and removes their stays.

> **Check.** You should see 2,423 encounters removed and 99,343 left.

In [ ]:
ENDED = [11, 13, 14, 19, 20, 21]   # the patient died or went to a hospice
for code in ENDED:
    print(code, codes["discharge_disposition_id"][code])
data = raw[~raw["discharge_disposition_id"].isin(ENDED)].copy()
print(len(raw) - len(data), "encounters removed,", len(data), "left")

**What this tells us.** We print the codes from the table instead of trusting a list from memory, so anybody can check the step.

The next cell makes the target: 1 for a return within 30 days.

> **Check.** You should see 11,314 encounters, 11.4%.

In [ ]:
TARGET = "readmitted_30"
data[TARGET] = (data["readmitted"] == "<30").astype(int)
print(data[TARGET].sum(), "encounters were followed by a readmission within 30 days:",
      f"{data[TARGET].mean():.1%}")

**What this tells us.** At random, about 11 calls in 100 would reach a patient who comes back. Every model must beat that.

## 3. Frame the problem

We predict at **discharge**, for one encounter. An input is allowed only if the team knows it on that day. Counts of visits in the year **before** the stay are fine; anything after discharge is a leak. `race`, `gender` and `payer_code` are **sensitive**: they stay out of the inputs and are used only for slice checks. Age is a medical risk factor, so it is an input, and we check every age band. ([Frame the problem on the page](https://learning.nextia-ai.com/courses/ml/m09/choose-a-model-for-readmissions/#frame-the-problem).)

The main diagnosis is an ICD-9 code. The next cell groups the codes as Strack et al. (2014) did, and tests the function on five known codes.

> **Check.** You should see `['circulatory', 'diabetes', 'respiratory', 'other', 'injury']`.

In [ ]:
GROUPS = [("circulatory", 390, 460, "785"), ("respiratory", 460, 520, "786"),
          ("digestive", 520, 580, "787"), ("injury", 800, 1000, None),
          ("musculoskeletal", 710, 740, None), ("genitourinary", 580, 630, "788"),
          ("neoplasms", 140, 240, None)]


def diagnosis_group(code):
    """The group of the main diagnosis (an ICD-9 code), as in Strack et al. (2014)."""
    if pd.isna(code):
        return "missing"
    if code.startswith(("V", "E")):
        return "other"                        # visits for other reasons, and external causes
    if code.startswith("250"):
        return "diabetes"
    number = float(code)
    for name, low, high, extra in GROUPS:
        if low <= number < high or (extra and code.startswith(extra)):
            return name
    return "other"


print([diagnosis_group(code) for code in ["428", "250.83", "786", "V57", "996"]])

**What this tells us.** Testing a function on a few known values takes one line and catches many mistakes.

The next cell translates the codes, joins the "unknown" codes, turns the age band into a number and fills the empty test results.

> **Check.** You should see five rows; the first one has `unknown`, `diabetes`, `5` and `not measured`.

In [ ]:
UNKNOWN = ["NULL", "Not Available", "Not Mapped", "Unknown/Invalid"]
for column, name in [("admission_type_id", "admission_type"), ("admission_source_id", "admission_source"),
                     ("discharge_disposition_id", "discharge_to")]:
    data[name] = data[column].map(codes[column]).replace(UNKNOWN, "unknown").fillna("unknown")
data["diagnosis_group"] = data["diag_1"].map(diagnosis_group)
data["age_years"] = data["age"].str.extract(r"(\d+)")[0].astype(int) + 5   # "[70-80)" becomes 75
data["specialty"] = data["medical_specialty"].fillna("missing")
tests = ["A1Cresult", "max_glu_serum"]
data[tests] = data[tests].fillna("not measured")
data[["admission_type", "discharge_to", "diagnosis_group", "age_years", "A1Cresult"]].head()

**What this tells us.** The codes are now words that a nurse can read, and age is a number, so "older" means "larger".

The next cell writes down the feature lists, and checks that no sensitive column is an input.

> **Check.** You should see `19 features; sensitive columns among them: set()`.

In [ ]:
CATEGORIES = ["admission_type", "admission_source", "discharge_to", "specialty", "diagnosis_group",
              "A1Cresult", "max_glu_serum", "insulin", "change", "diabetesMed"]
NUMBERS = ["age_years", "time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
           "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses"]
FEATURES = CATEGORIES + NUMBERS
SENSITIVE = ["race", "gender", "payer_code"]   # for slice checks only, never inputs
print(len(FEATURES), "features; sensitive columns among them:", set(SENSITIVE) & set(FEATURES))

**What this tells us.** The model sees 10 categories and 9 numbers. `set()` is an empty set: no sensitive column is an input.

## 4. Explore what the decision needs

The nurses' rule uses one fact: hospital stays in the past year. How strong is it?

> **Predict.** How much higher is the readmission rate with 5 or more earlier stays than with none?

In [ ]:
prior_stays = data["number_inpatient"].clip(upper=5)   # 5 means "5 or more"
data.groupby(prior_stays)[TARGET].agg(["size", "mean"]).round(3)

**What this tells us.** The rate climbs from 8.6% with no earlier stay to 37.1% with five or more: a good rule. But 66,245 stays had no earlier stay, and their rate is 8.6%, not zero. The rule cannot tell those patients apart.

Where the patient goes after the stay is the second thing a nurse knows at discharge. The next cell shows the destinations with at least 1,000 stays.

In [ ]:
by_discharge = data.groupby("discharge_to")[TARGET].agg(["size", "mean"])
by_discharge[by_discharge["size"] >= 1000].sort_values("mean", ascending=False).round(3)

**What this tells us.** Patients sent to a rehabilitation unit come back at 27.7%, three times as often as those who go home (9.3%). The rule does not use this fact, so a model that combines both may rank better.

> **Your turn.** Do emergency visits in the past year matter too? Make `had_emergency`, a True/False Series for encounters with at least one emergency visit (`number_emergency`). Then put the readmission rate of those encounters in `rate_with`, and of the others in `rate_without`, each rounded to 3 decimals.

In [ ]:
had_emergency = ...   # True for encounters with at least one emergency visit in the past year
rate_with = ...       # readmission rate (TARGET mean) of those encounters, rounded to 3 decimals
rate_without = ...    # readmission rate of the other encounters, rounded to 3 decimals
print(rate_with, rate_without)

In [ ]:
expect_hash('the two rates', None if ... in (rate_with, rate_without) else [rate_with, rate_without], '1d8d96f5f023be00')

## 5. Split by patient

The next cell counts the patients with more than one stay.

> **Check.** You should see 69,990 patients, 16,341 with more than one encounter, and 45,694 of 99,343 rows.

In [ ]:
encounters = data["patient_nbr"].value_counts()
print(encounters.size, "patients;", (encounters > 1).sum(), "of them have more than one encounter")
print("their encounters:", encounters[encounters > 1].sum(), "of", len(data), "rows")
print("most encounters for one patient:", encounters.max())

**What this tells us.** Almost half of the rows belong to repeat patients, and one patient has 40 stays. Two stays of the same person are not independent, so a split by row would put the same patient in train and test.

The next cell gives each **patient** a part: 60% train, 20% valid, 20% test, with a fixed seed. All the stays of a patient go to the same part.

> **Check.** You should see train 59,556, valid 20,179 and test 19,608 encounters, rates from 0.113 to 0.116, and 0 patients in more than one part.

In [ ]:
patients = data["patient_nbr"].unique()
rng = np.random.default_rng(0)
part_of = dict(zip(patients, rng.choice(["train", "valid", "test"], size=len(patients), p=[0.6, 0.2, 0.2])))
data["part"] = data["patient_nbr"].map(part_of)
train, valid, test = (data[data["part"] == name].copy() for name in ["train", "valid", "test"])
for name, part in [("train", train), ("valid", valid), ("test", test)]:
    print(f"{name}: {len(part)} encounters, {part['patient_nbr'].nunique()} patients,"
          f" readmitted {part[TARGET].mean():.3f}")
print("patients in more than one part:", (data.groupby("patient_nbr")["part"].nunique() > 1).sum())

**What this tells us.** The parts have the same readmission rate, and no patient is in two parts. From now on, the test set is sealed until the final test. The file has no dates, so a split by time is not possible; that is a limit.

To see what a row split would do, split the **train** set by row (the test set stays sealed).

> **Check.** You should see 4,046 patients in both parts and 4,915 of 11,912 check rows (41%).

In [ ]:
from sklearn.model_selection import train_test_split

fit_rows, check_rows = train_test_split(train, test_size=0.2, random_state=0)
seen = check_rows["patient_nbr"].isin(fit_rows["patient_nbr"])
print("patients in both parts:", check_rows.loc[seen, "patient_nbr"].nunique())
print(f"check rows of a patient that the model has seen: {seen.sum()} of {len(check_rows)} ({seen.mean():.0%})")

**What this tells us.** With a row split, 41% of the checked rows belong to patients that the model trained on. That score mixes "how well does the model know these patients?" with "how well does it predict new ones?". Only the second question matters to Nadia.

## 6. Baselines and the protocol

Every model gets the same **protocol**: the train set, the same 5 folds **by patient** (`StratifiedGroupKFold`), the same metric (average precision, AP), and the same pipeline with the 19 features. `min_frequency=100` joins categories with fewer than 100 rows into one "infrequent" column; `handle_unknown="infrequent_if_exist"` sends a new category there too. The [protocol diagram](https://learning.nextia-ai.com/courses/ml/m09/choose-a-model-for-readmissions/#one-protocol-for-every-model) is on the page.

Run the next cell. It prints nothing.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

folds = list(StratifiedGroupKFold(5, shuffle=True, random_state=0)
             .split(train, train[TARGET], groups=train["patient_nbr"]))


def build_pipeline(model, numbers=NUMBERS):
    """The same preparation for every model: one-hot categories (rare ones grouped) and scaled numbers."""
    prepare = ColumnTransformer([
        ("categories", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=100,
                                     sparse_output=False), CATEGORIES),
        ("numbers", StandardScaler(), numbers)])
    return Pipeline([("prepare", prepare), ("model", model)])

The next cell checks the folds.

> **Check.** You should see five folds of about 11,910 rows, rates 0.113 to 0.114, and 0 shared patients in each.

In [ ]:
for number, (fit, check) in enumerate(folds, 1):
    shared = set(train["patient_nbr"].iloc[fit]) & set(train["patient_nbr"].iloc[check])
    print(f"fold {number}: {len(check)} check rows, readmitted {train[TARGET].iloc[check].mean():.3f},"
          f" patients also in the fit rows: {len(shared)}")

**What this tells us.** Each fold checks about 11,900 rows with the same rate, and no patient is on both sides. Every model below uses exactly these `folds`.

The next cell scores two baselines on the folds and starts the log: **E01** gives every patient the same score (the floor), and **E02** is the nurses' rule as a score (stays in the past year; emergency visits break ties).

> **Check.** You should see E01 at 0.114 and E02 at 0.182.

In [ ]:
from sklearn.metrics import average_precision_score


def stays_rule(df):
    """The rule of thumb: most hospital stays in the past year first; emergency visits break ties."""
    return df["number_inpatient"] + df["number_emergency"].clip(upper=99) / 100


def rule_folds(score):
    """The AP of a fixed score (nothing to train) on each of the 5 check folds."""
    return np.array([average_precision_score(train[TARGET].iloc[check], score.iloc[check])
                     for _, check in folds])


log_experiment("E01", "Baseline: the same score for every patient", rule_folds(train["number_inpatient"] * 0.0),
               decision="baseline", reason="The floor: its AP is the readmission rate.")
log_experiment("E02", "Baseline: the prior-stays rule", rule_folds(stays_rule(train)),
               decision="baseline", reason="The rule of thumb that a model must beat.")
show_log()

**What this tells us.** The floor is the readmission rate, 0.114. The rule reaches 0.182. A model that cannot beat 0.182 is not worth building.

> **Your turn.** Would emergency visits alone make a better rule? Put `rule_folds(...)` of a score that is only `train["number_emergency"]` in `emergency_folds`. Do not log it.

In [ ]:
emergency_folds = ...   # rule_folds() of a score that is only number_emergency
print(emergency_folds)

In [ ]:
expect_hash('the mean AP of the emergency rule, rounded to 3 decimals', None if emergency_folds is ... else round(float(np.mean(emergency_folds)), 3), '13edbf90d092d8a1')

## 7. Four model families under one protocol

`tune()` runs a small grid search on the same folds and applies the **one-standard-error rule** inside each family: among the settings whose mean AP is within one standard error of the best, it keeps the first one, and each grid lists its simplest setting first. Run the next cell. It prints nothing.

In [ ]:
from sklearn.model_selection import GridSearchCV


def tune(pipeline, grid):
    """Score each setting of `grid` on the 5 patient folds. Keep the simplest setting whose mean is
    within one standard error of the best (each grid lists the simplest setting first)."""
    search = GridSearchCV(pipeline, grid, cv=folds, scoring="average_precision", refit=False)
    results = pd.DataFrame(search.fit(train[FEATURES], train[TARGET]).cv_results_)
    results["se"] = results["std_test_score"] / np.sqrt(len(folds))
    best = results["mean_test_score"].idxmax()
    bar = results.loc[best, "mean_test_score"] - results.loc[best, "se"]
    chosen = results.index[results["mean_test_score"] >= bar][0]
    print(results[["params", "mean_test_score", "std_test_score"]].round(4).to_string())
    print("one-SE bar:", round(bar, 4), "| chosen:", results.loc[chosen, "params"])
    pipeline.set_params(**results.loc[chosen, "params"])
    return np.array([results.loc[chosen, f"split{i}_test_score"] for i in range(len(folds))])

**Logistic regression.** `C` is the strength of the penalty: a small `C` is a strong penalty and a simpler model. It takes a few seconds.

> **Check.** You should see the three settings between 0.2100 and 0.2116, and `chosen: {'model__C': 0.01}`.

In [ ]:
from sklearn.linear_model import LogisticRegression

logistic = build_pipeline(LogisticRegression(max_iter=1000))
scores = tune(logistic, {"model__C": [0.01, 0.1, 1.0]})
log_experiment("E03", "Logistic regression; C tuned over 0.01, 0.1, 1", scores, decision="keep",
               reason="Well above the rule; the strongest penalty is within one SE of the best.")

**What this tells us.** The three settings differ by 0.0016, far less than one standard error (about 0.0045), so the rule keeps the simplest. Logistic regression beats the rule clearly: 0.210 against 0.182.

**Decision tree.** The setting is the depth, with at least 100 stays in each leaf.

> **Check.** You should see `chosen: {'model__max_depth': 6}` with 0.2020.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree = build_pipeline(DecisionTreeClassifier(min_samples_leaf=100, random_state=0))
scores = tune(tree, {"model__max_depth": [4, 6, 8, 10]})
log_experiment("E04", "Decision tree, min leaf 100; depth tuned over 4, 6, 8, 10", scores, decision="keep",
               reason="Above the rule, below logistic regression.")

**What this tells us.** Depth 6 and 8 tie, and depth 10 gets worse. The best tree (0.202) is below logistic regression (0.210).

**Random forest.** 100 trees, to keep the notebook fast. The setting is the minimum leaf size: larger leaves make simpler trees. It takes 10 to 40 seconds.

> **Check.** You should see `chosen: {'model__min_samples_leaf': 20}` with 0.2206.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest = build_pipeline(RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=0))
scores = tune(forest, {"model__min_samples_leaf": [50, 20, 10]})
log_experiment("E05", "Random forest, 100 trees; min leaf tuned over 50, 20, 10", scores, decision="keep",
               reason="Above logistic regression on the folds.")

**What this tells us.** A leaf of 10 scores best (0.2237), but 20 is within one standard error, so the rule keeps it. The forest is the first model above logistic regression.

**Gradient boosting.** 100 trees, always all of them (`early_stopping=False`). The setting is the depth of each tree. It takes 10 to 40 seconds.

> **Check.** You should see `chosen: {'model__max_depth': 2}` with 0.2188.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

boosting = build_pipeline(HistGradientBoostingClassifier(early_stopping=False, random_state=0))
scores = tune(boosting, {"model__max_depth": [2, 3, None]})
log_experiment("E06", "Gradient boosting, 100 trees; depth tuned over 2, 3, no limit", scores, decision="keep",
               reason="The best family on the folds, with the forest.")

**What this tells us.** Depth 3 is best (0.2226); depth 2 just reaches the bar, so the rule keeps trees of depth 2.

### Compare the families

The next cell puts the chosen setting of every family side by side, with the standard error of each mean.

> **Predict.** Is logistic regression within one standard error of the best family?

In [ ]:
compare = pd.DataFrame({id: FOLD_SCORES[id] for id in ["E02", "E03", "E04", "E05", "E06"]},
                       index=[f"fold {i}" for i in range(1, 6)])
summary = pd.DataFrame({"mean": compare.mean(), "se": compare.std(ddof=0) / np.sqrt(len(folds))})
best = summary["mean"].idxmax()
bar = summary.loc[best, "mean"] - summary.loc[best, "se"]
print(summary.round(4).T)
print("best:", best, "| one-SE bar:", round(bar, 4), "| within one SE:", summary.index[summary["mean"] >= bar].tolist())
print("boosting minus logistic, per fold:", (compare["E06"] - compare["E03"]).round(4).tolist())

The next cell draws the same comparison.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 3))
ax.errorbar(summary["mean"], range(len(summary)), xerr=summary["se"], fmt="o", capsize=4)
ax.axvline(bar, color="grey", linestyle="--", label="one-SE bar")
ax.set_yticks(range(len(summary)), ["E02 rule", "E03 logistic", "E04 tree", "E05 forest", "E06 boosting"])
ax.set_xlabel("CV average precision (mean ± one standard error)")
ax.legend()
plt.show()

**What this tells us.** The forest has the best mean (0.2206), and the bar is 0.2162. Only the forest and boosting reach it: they tie, and boosting is far smaller, so it is the **challenger**. Logistic regression (0.2100) is below the bar, about two standard errors behind, and boosting is higher on all five folds. Here the complex model wins by more than the noise. Is the gain worth it on the decision? Sections 8 and 9 answer that.

> **Your turn.** Apply the one-standard-error rule yourself, from `summary`. Put the best mean in `best_mean`, its standard error in `one_se`, the bar `best_mean - one_se` rounded to 4 decimals in `bar`, and in `logistic_within` whether E03 reaches the bar (True or False).

In [ ]:
best_mean = ...        # the mean CV AP of the best model in summary
one_se = ...           # its standard error
bar = ...              # best_mean - one_se, rounded to 4 decimals
logistic_within = ...  # True if logistic regression (E03) reaches the bar
print(bar, logistic_within)

In [ ]:
expect_hash('the bar and the answer', None if ... in (bar, logistic_within) else [bar, logistic_within], '80693ee517703ba9')

### Two experiments that you must reject

**E07.** How much would folds by row change the score? The next cell scores the same boosting pipeline with ordinary folds by row. It takes a few seconds.

> **Check.** You should see a mean of 0.2193 by row and 0.2188 by patient.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

row_folds = StratifiedKFold(5, shuffle=True, random_state=0)
row_scores = cross_val_score(boosting, train[FEATURES], train[TARGET], cv=row_folds, scoring="average_precision")
print("boosting, folds by row:    ", row_scores.round(4), round(row_scores.mean(), 4))
print("boosting, folds by patient:", FOLD_SCORES["E06"].round(4), round(FOLD_SCORES["E06"].mean(), 4))
log_experiment("E07", "E06 scored with folds by row (patients in both parts)", row_scores, decision="reject",
               reason="Not a valid protocol: it scores patients that the model has seen.")

**What this tells us.** On this data, the row split changes the score by only 0.0005. The inputs describe the stay, not the person, so the model cannot recognise a patient. With an input that is fixed for a person, the same mistake gives a large, false gain. We split by patient because of what it measures.

**E08.** Dev counts each patient's stays in the whole file and adds the count as a feature. This is the [failure case on the page](https://learning.nextia-ai.com/courses/ml/m09/choose-a-model-for-readmissions/#failure-case-a-count-that-looks-into-the-future).

> **Predict.** The count includes stays **after** the one we predict. What will it do to the score?

In [ ]:
in_file = data["patient_nbr"].map(data["patient_nbr"].value_counts())   # all encounters, also later ones
train["encounters_in_file"] = in_file.loc[train.index]
leaky = build_pipeline(LogisticRegression(C=0.01, max_iter=1000), numbers=NUMBERS + ["encounters_in_file"])
leaky_scores = cross_val_score(leaky, train[FEATURES + ["encounters_in_file"]], train[TARGET], cv=folds,
                               scoring="average_precision")
print("with encounters_in_file:", leaky_scores.round(4), round(leaky_scores.mean(), 4))
print("without (E03):          ", FOLD_SCORES["E03"].round(4), round(FOLD_SCORES["E03"].mean(), 4))
log_experiment("E08", "E03 plus encounters_in_file (counts later encounters too)", leaky_scores,
               decision="reject", reason="A leak: at discharge, nobody knows the later encounters.")

**What this tells us.** The AP jumps from 0.210 to 0.244, more than any model family gave. A patient who comes back has, by definition, another stay in the file, so the feature partly contains the answer. At discharge, nobody knows the future stays. Reject it, and keep the reason in the log.

## 8. Metric and operating point

AP chose the families. The decision uses only the top 10%, so the decision metric is **precision@10%**: the share of called patients who would come back. The next cell fits each tuned pipeline on the whole train set and scores the validation set. It takes 10 to 30 seconds.

> **Check.** You should see precision@10% of 0.240 (rule), 0.261 (logistic), 0.251 (tree), 0.258 (forest) and 0.268 (boosting).

In [ ]:
from sklearn.metrics import roc_auc_score


def precision_at(y, score, share=0.10):
    """The precision among the `share` of patients with the highest scores: the patients the team calls."""
    y, score = np.asarray(y), np.asarray(score)
    top = np.argsort(-score, kind="stable")[: int(share * len(y))]
    return y[top].mean()


models = {"E03 logistic": logistic, "E04 tree": tree, "E05 forest": forest, "E06 boosting": boosting}
valid_scores = {"E02 rule": stays_rule(valid).to_numpy()}
for name, pipeline in models.items():
    valid_scores[name] = pipeline.fit(train[FEATURES], train[TARGET]).predict_proba(valid[FEATURES])[:, 1]
table = pd.DataFrame({name: {"AP": average_precision_score(valid[TARGET], score),
                             "ROC AUC": roc_auc_score(valid[TARGET], score),
                             "precision@10%": precision_at(valid[TARGET], score)}
                      for name, score in valid_scores.items()}).T
table.round(3)

**What this tells us.** The order by AP is the same as in CV. But at the top 10% the differences shrink: all four models reach 0.251 to 0.268, and the rule 0.240. A difference in AP does not always reach the decision.

The next cell turns the precision into patients: the validation set has 20,179 discharges, so 2,017 calls.

> **Check.** You should see 484 (rule), 526 (logistic) and 540 (boosting) of 2,273.

In [ ]:
calls = int(0.10 * len(valid))
for name in ["E02 rule", "E03 logistic", "E06 boosting"]:
    reached = round(precision_at(valid[TARGET], valid_scores[name]) * calls)
    print(f"{name:13s} {calls} calls reach {reached} of the {valid[TARGET].sum()} patients who return")

**What this tells us.** Logistic regression reaches 42 more patients than the rule, and boosting 14 more than logistic regression. Are these gaps real, or luck?

### Is the gain real? A bootstrap interval

`bootstrap_gap()` draws a new validation set of the same size 1,000 times, by **patient**, and measures the gap each time; the middle 95% is the interval, as in [Compare alternatives](https://learning.nextia-ai.com/courses/math/m05/compare-alternatives/#an-interval-for-the-difference) in the mathematics course. Run the next cell. It prints nothing.

In [ ]:
def bootstrap_gap(metric, a, b, repeats=1000):
    """metric(b) - metric(a) on 1,000 resampled validation sets. It resamples patients, not rows,
    because the encounters of one patient are not independent."""
    rows_of = list(valid.groupby("patient_nbr").indices.values())
    y = valid[TARGET].to_numpy()
    rng = np.random.default_rng(0)
    gaps = []
    for _ in range(repeats):
        rows = np.concatenate([rows_of[i] for i in rng.integers(0, len(rows_of), len(rows_of))])
        gaps.append(metric(y[rows], b[rows]) - metric(y[rows], a[rows]))
    return np.array(gaps)


def show_gap(name, metric, a, b):
    gaps = bootstrap_gap(metric, a, b)
    low, high = np.percentile(gaps, [2.5, 97.5])
    observed = metric(valid[TARGET], b) - metric(valid[TARGET], a)
    print(f"{name}: gap {observed:+.4f}, 95% interval {low:+.4f} to {high:+.4f}")

The next cell measures three gaps. It takes about 10 to 20 seconds.

> **Predict.** Which of the three intervals includes zero?

In [ ]:
logistic_score, boosting_score = valid_scores["E03 logistic"], valid_scores["E06 boosting"]
show_gap("boosting - logistic, AP           ", average_precision_score, logistic_score, boosting_score)
show_gap("boosting - logistic, precision@10%", precision_at, logistic_score, boosting_score)
show_gap("logistic - rule, precision@10%    ", precision_at, valid_scores["E02 rule"], logistic_score)

**What this tells us.** Boosting ranks better overall (AP gap +0.0085, interval +0.004 to +0.013). At Nadia's capacity, the gap is +0.007 with an interval from −0.006 to +0.017: we cannot tell the two apart. Logistic regression clearly beats the rule (+0.021, interval +0.006 to +0.036).

### Is the gain worth the complexity?

Use the checklist of [When a simple model is better](https://learning.nextia-ai.com/courses/ml/m07/when-a-simple-model-is-better/#a-decision-checklist). The nurses need a reason for each call, and clinical governance must approve the tool. Boosting wins by more than one standard error in ranking, but at the capacity the gain is within the noise. The next cell measures the cost in size and speed. The times depend on your computer.

> **Check.** You should see about 12 KB for logistic regression, about 15,000 KB for the forest and about 79 KB for boosting.

In [ ]:
import io
import time

import joblib


def size_and_speed(pipeline):
    """The size of the saved pipeline (KB) and the time to score the validation set (seconds)."""
    buffer = io.BytesIO()
    joblib.dump(pipeline, buffer)
    start = time.perf_counter()
    pipeline.predict_proba(valid[FEATURES])
    return round(buffer.tell() / 1024), round(time.perf_counter() - start, 3)


for name in ["E03 logistic", "E05 forest", "E06 boosting"]:
    print(name, "size KB, seconds to score valid:", size_and_speed(models[name]))

**What this tells us.** All three are fast. The real cost of boosting is that its score comes from 100 trees: no exact reasons for each call, and harder to monitor and review. **Decision:** logistic regression (E03) for a pilot, boosting (E06) as the documented challenger.

The next cell writes the decisions and the valid AP of the two finalists into the log.

> **Check.** You should see eight rows; E03 is `keep`, with valid AP 0.206.

In [ ]:
log_experiment("E03", "Logistic regression; C tuned over 0.01, 0.1, 1", FOLD_SCORES["E03"],
               valid_ap=table.loc["E03 logistic", "AP"], decision="keep",
               reason="Chosen: at 10% capacity it ties boosting, and each call gets reasons.")
log_experiment("E06", "Gradient boosting, 100 trees; depth tuned over 2, 3, no limit", FOLD_SCORES["E06"],
               valid_ap=table.loc["E06 boosting", "AP"], decision="reject",
               reason="Better AP beyond one SE, but no clear gain at 10%; kept as the challenger.")
log_experiment("E04", "Decision tree, min leaf 100; depth tuned over 4, 6, 8, 10", FOLD_SCORES["E04"],
               decision="reject", reason="Below logistic regression on every fold.")
log_experiment("E05", "Random forest, 100 trees; min leaf tuned over 50, 20, 10", FOLD_SCORES["E05"],
               decision="reject", reason="Ties boosting on the folds; bigger and slower than boosting.")
show_log()

**What this tells us.** The log tells the whole story in eight rows, including the two traps. Anyone can see what was tried, how it scored and why it was kept or rejected.

### The threshold and a reason for each call

The next cell sets the threshold on valid: the score of the 2,017th patient.

> **Check.** You should see `threshold: 0.18 | called: 2017 of 20179`.

In [ ]:
score = valid_scores["E03 logistic"]
threshold = np.sort(score)[::-1][calls - 1]
valid["score"] = score
valid["called"] = valid["score"] >= threshold
print("threshold:", round(threshold, 3), "| called:", valid["called"].sum(), "of", len(valid))

**What this tells us.** A patient with a score of 0.18 or more gets a call.

The next cell splits the score of a typical called patient into contributions, as in [Explain predictions](https://learning.nextia-ai.com/courses/ml/m07/explain-predictions/).

> **Check.** You should see a score of 0.226, a patient in their 80s with 4 earlier stays, and `number_inpatient` with a contribution of 0.89.

In [ ]:
def contributions(pipeline, patient):
    """Coefficient times prepared value, for each column of one patient (as in Explain predictions)."""
    prepare, model = pipeline.named_steps["prepare"], pipeline.named_steps["model"]
    values = np.asarray(prepare.transform(patient[FEATURES]))[0]
    names = [name.split("__", 1)[1][:40] for name in prepare.get_feature_names_out()]
    table = pd.DataFrame({"value": values, "coefficient": model.coef_[0]}, index=names)
    table["contribution"] = table["value"] * table["coefficient"]
    return table


called = valid[valid["called"]].sort_values("score")
patient = called.iloc[[len(called) // 2]]            # a typical called patient: the middle of the list
print("score:", round(patient["score"].iloc[0], 3), "| readmitted:", patient[TARGET].iloc[0])
print(patient[["age", "number_inpatient", "number_emergency", "num_medications"]].to_string(index=False))
reasons = contributions(logistic, patient)
print(reasons[reasons["value"] != 0].sort_values("contribution", ascending=False).head(5).round(2))

**What this tells us.** Four hospital stays in the past year add +0.89 to the model's sum, ten times more than any other fact. The nurse can open the call with it. A contribution describes the model, not the cause.

> **Your turn (change one thing).** Nadia's team may grow to call 20% of patients. Put `precision_at(valid[TARGET], valid_scores[name], share=0.20)`, rounded to 3 decimals, for `"E02 rule"`, `"E03 logistic"` and `"E06 boosting"` in a dictionary `at_20`. Before you run it, predict: does the gap between boosting and logistic regression grow?

In [ ]:
at_20 = ...   # {"E02 rule": ..., "E03 logistic": ..., "E06 boosting": ...}: precision_at(..., share=0.20) on valid, 3 decimals
print(at_20)

In [ ]:
expect_hash('the three precisions at 20%', None if at_20 is ... else at_20, '3f4abec7fa272a65')

## 9. Errors and slices

The next cell defines `slice_table()` and checks the model in each age band (the bands under 40 are joined, because they are small).

> **Check.** You should see seven bands; the AUC falls from 0.762 under 40 to 0.559 for 90 or more.

In [ ]:
def slice_table(df, column):
    """Per group: encounters, readmission rate, share called, precision and recall of the calls, AUC."""
    groups = df.groupby(column, observed=True)
    return pd.DataFrame({
        "encounters": groups.size(),
        "readmitted": groups[TARGET].mean(),
        "called": groups["called"].mean(),
        "precision": groups.apply(lambda g: g.loc[g["called"], TARGET].mean()),
        "recall": groups.apply(lambda g: g.loc[g[TARGET] == 1, "called"].mean()),
        "AUC": groups.apply(lambda g: roc_auc_score(g[TARGET], g["score"]) if g[TARGET].nunique() == 2 else np.nan),
    }).round(3)


valid["age_band"] = valid["age"].replace({"[0-10)": "[0-40)", "[10-20)": "[0-40)", "[20-30)": "[0-40)",
                                          "[30-40)": "[0-40)"})
slice_table(valid, "age_band")

The next cell draws the AUC by age band.

In [ ]:
import matplotlib.pyplot as plt

ages = slice_table(valid, "age_band")
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(ages.index.astype(str), ages["AUC"])
ax.axhline(0.5, color="grey", linestyle="--", label="guessing")
ax.set_ylim(0.4, 0.8)
ax.set_xlabel("age band")
ax.set_ylabel("ROC AUC in the band")
ax.legend()
plt.show()

**What this tells us.** The model ranks younger patients best and gets weaker with every decade. It calls the oldest patients most often (17.2%), but there its calls are the least precise (0.143). For them, the nurses' judgement matters more.

The next cell checks the sensitive columns. They are not inputs; the question is whether the model still treats the groups differently.

> **Check.** You should see three tables: race, gender and payer.

In [ ]:
valid["race_group"] = valid["race"].fillna("missing")
valid["payer_group"] = valid["payer_code"].fillna("missing").where(
    valid["payer_code"].isin(["MC", "HM", "SP", "BC", "MD"]) | valid["payer_code"].isna(), "other")
for column in ["race_group", "gender", "payer_group"]:
    print(slice_table(valid, column).to_string(), end="\n\n")

**What this tells us.** The two large race groups have the same rate and are called at the same rate, but the calls are less precise for African American patients (0.237 against 0.269), with lower recall. With 3,849 stays, that gap is worth watching. Women and men have the same rate and almost the same recall. The payer group MC (probably Medicare, older patients) is called most. The smaller groups are too small to judge. Report these tables and watch them in the pilot.

> **Your turn.** Put the share of encounters called (`valid["called"]`) for `"Female"` and for `"Male"`, each rounded to 3 decimals, in a list `gender_called`: `[female, male]`.

In [ ]:
gender_called = ...   # the share of encounters called (valid["called"]) for Female and for Male, 3 decimals: [female, male]
print(gender_called)

In [ ]:
expect_hash('the share called for women and men', None if gender_called is ... else gender_called, 'd98c53e3864873e9')

The next cell asks who the model misses.

> **Check.** You should see 1,116 of 2,273 (49%) with no earlier stay, and a recall of 0.058 among them.

In [ ]:
returned = valid[valid[TARGET] == 1]
first = returned["number_inpatient"] == 0
print(f"patients who return: {len(returned)}; with no stay in the past year: {first.sum()} ({first.mean():.0%})")
print(f"recall among them: {returned.loc[first, 'called'].mean():.3f};"
      f" among the others: {returned.loc[~first, 'called'].mean():.3f}")

**What this tells us.** Half of the patients who come back had no hospital stay in the past year, and the model calls only 5.8% of them. The model, like the rule, mostly finds patients who come back often. That is a limit of the data, not of the model family.

## 10. The final test, once

Everything is decided: logistic regression with `C = 0.01`, fitted on train, and the threshold 0.18. The next cell opens the sealed test set, once. Whatever it shows, we do not change the model.

> **Check.** You should see 1,870 called (9.5%), precision 0.270, recall 0.221, ROC AUC 0.663, and the rule at 0.247.

In [ ]:
test_score = logistic.predict_proba(test[FEATURES])[:, 1]
called = test_score >= threshold
returned = test[TARGET].to_numpy() == 1
print(f"test: {len(test)} encounters, {returned.sum()} returned within 30 days")
print(f"called with the valid threshold: {called.sum()} ({called.mean():.1%})")
print(f"precision {returned[called].mean():.3f}, recall {called[returned].mean():.3f}")
print(f"AP {average_precision_score(returned, test_score):.3f}, ROC AUC {roc_auc_score(returned, test_score):.3f},"
      f" precision@10% {precision_at(returned, test_score):.3f}")
print(f"the rule on test: precision@10% {precision_at(returned, stays_rule(test)):.3f}")

**What this tells us.** On patients that no step has seen, the model behaves as on valid. About 27 calls in 100 reach a patient who comes back, against 11 at random and 25 with the rule. The share called (9.5%) is close to the planned 10%, so the team should count the calls and adjust.

## 11. Recommendations, limits and your portfolio

The [page](https://learning.nextia-ai.com/courses/ml/m09/choose-a-model-for-readmissions/#recommendations) has the full recommendations for Nadia and Felix, the limits, and a README template for your repository. In short:

- Start a pilot with logistic regression at the threshold 0.18, show the nurses the main reasons for each call, and let them add patients the list misses.
- Measure the pilot (model against rule), watch the slices every month, and test the boosting challenger again on the new data.
- Be honest about the ceiling: ROC AUC about 0.67. Four families end within 0.02 AP of each other, so better data would help more than a better algorithm.

To make it a portfolio piece, put this notebook (run, with its outputs), `experiment_log.csv`, a README, a model card and the dataset card with its attribution in a public repository.

## Check your understanding and recap

Answer the **knowledge checks** on the [case-study page](https://learning.nextia-ai.com/courses/ml/m09/choose-a-model-for-readmissions/#knowledge-check).

You prepared real hospital data, split it by patient, compared four model families under one protocol with an experiment log, and chose with the one-standard-error rule and a bootstrap interval. Boosting ranked better, but not at the team's capacity, so the readable model won. Next, a case study without a target: segments of online-shop customers.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Segment online-shop customers](https://learning.nextia-ai.com/courses/ml/m09/segment-online-shop-customers/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m09/choose-a-model-for-readmissions/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The data is real, de-identified hospital data (Clore et al., UCI, CC BY 4.0).